## Version 1 — GeoSteerNet (SDF-based geological boundary detection)

**Credits:**
- [hengck23 — multi-trajectory prediction (MTP) with deep CNN for welllog inversion](https://www.kaggle.com/code/hengck23/rogii-cnn-mtp-demo) — data preparation ideas and validation setup
- [Competition discussion #699853](https://www.kaggle.com/competitions/rogii-wellbore-geology-prediction/discussion/699853) — geological inversion framing and boundary function formulation

**Training notebook:** [rogii-cnn-mtp-train v325493489](https://www.kaggle.com/code/medali1992/rogii-cnn-mtp-train?scriptVersionId=325493489)

**CV results (row RMSE, 5-fold GroupKFold):**

| Fold | row RMSE |
|------|----------|
| 0    | 7.2392   |
| 1    | 8.0968   |
| 2    | 6.8485   |
| 3    | 7.2994   |
| 4    | 8.0322   |
| **Mean ± Std** | **7.503 ± 0.484** |

---

- **Misfit heatmap construction:** For each well, we build a 2D image `H(z, x) = f(z) − g(x)` where rows index typewell TVT depth and columns index compressed lateral MD segments. The true geological boundary — TVT — is the zero-crossing contour of this heatmap.

- **Signed Distance Function (SDF) supervision:** Rather than regressing TVT directly, the model is trained to predict the signed distance from every heatmap cell to the true boundary. This gives a rich, geometry-aware training signal across all 64×24 pixels rather than just the 24 boundary points.

- **History image encoding:** The known pre-PS trajectory is rendered as an anti-aliased line in a second 64×24 image, pixel-aligned with the heatmap. This lets the model directly relate where the boundary has been to where the GR matching pattern suggests it continues.

- **U-Net architecture (GeoSteerNet):** A residual encoder-decoder with skip connections processes the stacked (heatmap, history) 2-channel input. The bottleneck at spatial resolution 8×3 integrates global context; skip connections restore fine-grained boundary precision in the decoder.

- **Proximity-weighted loss:** The MSE loss is weighted by `exp(−|SDF_true| / 5)`, giving higher weight to cells near the boundary so the model focuses on getting the zero-crossing location right rather than fitting distant distance values.

- **Offset augmentation:** During training the PS anchor is shifted by ±4 compressed segments, synthetically generating samples at different lateral positions and teaching the model about the full shape of the matching landscape beyond the true PS location.

- **DDP training:** Two T4 GPUs via `torchrun --nproc_per_node=2` with 5-fold `GroupKFold` cross-validation, `CosineAnnealingWarmRestarts` scheduler, and rank-0-only validation to avoid `DistributedSampler` padding artifacts.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 0 — Directory Setup
# Run this cell FIRST before any %%writefile cell.
# ─────────────────────────────────────────────────────────────────────────────

import os
DATA_VERSION = "1"   # bump when features/NPZ changes


dirs = [
    "/kaggle/working/src",           # all importable modules live here
    "/kaggle/working/checkpoints",   # model checkpoints, one per fold
    "/kaggle/working/oof",           # out-of-fold prediction CSVs
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"✓ {d}")

# Write an empty __init__.py so Python treats src/ as a package.
init_path = "/kaggle/working/src/__init__.py"
with open(init_path, "w") as f:
    f.write("# auto-generated by setup cell\n")
print(f"✓ {init_path}")

# Confirm the sequences file exists before going further
SEQUENCES_DIR = f"/kaggle/input/datasets/medali1992/rogii-data-prep/sequences_v{DATA_VERSION}/"
if os.path.isdir(SEQUENCES_DIR):
    n_files = len([f for f in os.listdir(SEQUENCES_DIR) if f.endswith(".npz")])
    print(f"✓ Sequences directory found: {SEQUENCES_DIR}  ({n_files} wells)")
else:
    print(f"✗ Sequences directory NOT found: {SEQUENCES_DIR}")
    print("  Run prepare_sequences.py first, then re-run this notebook.")

print("\nAll directories ready — safe to run %%writefile cells now.")



# Config

In [ ]:
%%writefile /kaggle/working/src/config.py
"""
src/config.py  —  SDF pipeline
================================
Central configuration for the GeoSteerNet SDF pipeline.
"""

import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist


# ─────────────────────────────────────────────────────────────────────────────
# EXPERIMENT IDENTITY
# ─────────────────────────────────────────────────────────────────────────────
DATA_VERSION = "1"
RUN_VERSION  = "1"
PROJECT_NAME = "rogii-wellbore-geology-sdf"


# ─────────────────────────────────────────────────────────────────────────────
# DATA ROOT DISCOVERY
# ─────────────────────────────────────────────────────────────────────────────

def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    for root in candidates:
        if (root / "train").is_dir():
            return root.resolve()
    raise FileNotFoundError(
        "Could not find competition data. Tried:\n"
        + "\n".join(f"  {c}" for c in candidates)
    )


_DATA_ROOT = find_data_root()


# ─────────────────────────────────────────────────────────────────────────────
# CENTRAL CONFIG CLASS
# ─────────────────────────────────────────────────────────────────────────────

class CFG:
    # ── Data ──────────────────────────────────────────────────────────────────
    DATA_ROOT  = str(_DATA_ROOT)
    TRAIN_DIR  = str(_DATA_ROOT / "train")

    # ── Output ────────────────────────────────────────────────────────────────
    MODEL_DIR  = "/kaggle/working/checkpoints/"
    OOF_DIR    = "/kaggle/working/oof/"

    # ── Cross-validation ──────────────────────────────────────────────────────
    N_FOLDS    = 5

    # ── SDF pipeline (GeoSteerNet) ────────────────────────────────────────────
    SDF_BASE_CH       = 32
    SDF_TRAIN_OFFSETS = list(range(-4, 5, 2))   # [-4, -2, 0, 2, 4]
    SDF_VALID_OFFSETS = [0]

    # ── Training dynamics ─────────────────────────────────────────────────────
    EPOCHS        = 60
    BATCH_SIZE    = 16
    GRAD_ACC      = 1
    LEARNING_RATE = 2e-3
    WEIGHT_DECAY  = 1e-4
    GRAD_CLIP     = 1.0
    USE_AMP       = False
    MIN_LR        = 1e-5
    LR_T_0        = 15
    LR_T_MULT     = 1

    # ── Early stopping ────────────────────────────────────────────────────────
    PATIENCE   = 30

    # ── Reproducibility ───────────────────────────────────────────────────────
    SEED       = 42

    # ── Fast debug mode ───────────────────────────────────────────────────────
    FAST_DEBUG        = bool(int(os.environ.get("FAST_DEBUG", "0")))
    MAX_TRAIN_WELLS   = 24 if FAST_DEBUG else None
    _N_FOLDS_OVERRIDE = 2  if FAST_DEBUG else None
    _EPOCHS_OVERRIDE  = 1  if FAST_DEBUG else None

    # ── Generic getter ────────────────────────────────────────────────────────
    @classmethod
    def get(cls, key: str, default=None):
        return getattr(cls, key, default)

    @classmethod
    def apply_debug_overrides(cls):
        if cls.FAST_DEBUG:
            cls.N_FOLDS = cls._N_FOLDS_OVERRIDE
            cls.EPOCHS  = cls._EPOCHS_OVERRIDE
            print("[CFG] FAST_DEBUG=True — reduced folds/epochs/wells")

    @staticmethod
    def checkpoint_name(fold: int, epoch: int, rmse: float,
                        run_name: str = "") -> str:
        suffix = f"_{run_name}" if run_name else ""
        return (
            f"rogii_sdf_v{RUN_VERSION}"
            f"_fold{fold}"
            f"_ep{epoch:02d}"
            f"_rmse{rmse:.3f}"
            f"{suffix}.pth"
        )


# ─────────────────────────────────────────────────────────────────────────────
# DISTRIBUTED TRAINING HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def setup_ddp(rank: int, world_size: int) -> None:
    torch.cuda.set_device(rank)
    dist.init_process_group(
        backend="nccl",
        rank=rank,
        world_size=world_size,
        device_id=torch.device(f"cuda:{rank}"),
    )


def cleanup_ddp() -> None:
    dist.barrier()
    dist.destroy_process_group()


# ─────────────────────────────────────────────────────────────────────────────
# REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed: int = CFG.SEED) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark     = True


# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT DIRECTORY SETUP
# ─────────────────────────────────────────────────────────────────────────────

def make_output_dirs() -> None:
    for d in [CFG.MODEL_DIR, CFG.OOF_DIR]:
        os.makedirs(d, exist_ok=True)
    print(f"[CFG] Data root   : {CFG.DATA_ROOT}")
    print(f"[CFG] Train dir   : {CFG.TRAIN_DIR}")
    print(f"[CFG] Checkpoints : {CFG.MODEL_DIR}")
    print(f"[CFG] OOF dir     : {CFG.OOF_DIR}")

# Dataset

In [ ]:
%%writefile /kaggle/working/src/dataset.py
"""
src/dataset.py
==============
Dataset for the SDF-based GeoSteerNet pipeline.

Replaces the sequence-based WellSequenceDataset with GeoSteerDataset,
which builds (heatmap, history, sdf, masks) image samples from each well.
The DataLoader interface — collate, make_loader — is kept identical to
the previous version so engine.py changes are minimal.
"""

import sys
import os
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.ndimage import distance_transform_edt
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from src.config import CFG

# ─────────────────────────────────────────────────────────────────────────────
# Constants — geometry of the heatmap image
# ─────────────────────────────────────────────────────────────────────────────

S            = 32   # compression: 1 segment = S consecutive 1-ft samples
H_BEFORE_PS  = 8    # lateral segments before PS  (known history)
H_AFTER_PS   = 16   # lateral segments after  PS  (prediction zone)
H_TOTAL      = H_BEFORE_PS + H_AFTER_PS   # 24 columns
T_HALF       = 32   # typewell rows above and below the PS anchor
T_TOTAL      = T_HALF * 2                 # 64 rows

KAGGLE_DIR = CFG.DATA_ROOT


# ─────────────────────────────────────────────────────────────────────────────
# Well-level loading and preprocessing
# ─────────────────────────────────────────────────────────────────────────────

def load_well_data(well_id: str, split: str = "train") -> dict:
    """
    Load and preprocess one well's CSV data.
    Returns a dict of clean numpy arrays.
    Called once per well; result cached in GeoSteerDataset.__init__.
    """
    base = f"{KAGGLE_DIR}/{split}/{well_id}"
    h = pd.read_csv(f"{base}__horizontal_well.csv")
    t = pd.read_csv(f"{base}__typewell.csv")

    # Horizontal GR — interpolate gaps then smooth
    h_gr_raw    = h["GR"].interpolate(method="linear").bfill().ffill().values
    h_gr_smooth = savgol_filter(
        h_gr_raw, window_length=101, polyorder=2
    ).astype(np.float32)

    # TVT is present in training wells but absent in test wells
    # For test wells we use TVT_input as the known portion
    h_tvt_input = h["TVT_input"].values.astype(np.float32)
    if "TVT" in h.columns:
        h_tvt_true = h["TVT"].values.astype(np.float32)
    else:
        # Test wells: fill known portion from TVT_input, rest stays NaN
        # sliding_window_predict will fill the NaN region with predictions
        h_tvt_true = h_tvt_input.copy()

    # PS = last index where TVT_input is not NaN
    known_mask = ~np.isnan(h_tvt_input)
    ps_idx     = int(np.flatnonzero(known_mask)[-1]) if known_mask.any() else 0

    # Typewell
    t_gr  = t["GR"].values.astype(np.float32)
    t_tvt = t["TVT"].values.astype(np.float32)

    return {
        "well_id"     : well_id,
        "h_gr_smooth" : h_gr_smooth,
        "h_tvt_true"  : h_tvt_true,
        "h_tvt_input" : h_tvt_input,
        "ps_idx"      : ps_idx,
        "t_gr"        : t_gr,
        "t_tvt"       : t_tvt,
    }


# ─────────────────────────────────────────────────────────────────────────────
# Sample-level construction
# ─────────────────────────────────────────────────────────────────────────────

def build_sample(well_data: dict, offset: int = 0, step: int = S) -> dict:
    """
    Build one (heatmap, history, sdf, masks, target) training sample
    from a preprocessed well, with the PS anchor shifted by `offset`
    compressed segments.

    Raises ValueError if the shifted PS falls outside safe bounds.
    """
    h_gr   = well_data["h_gr_smooth"]
    h_tvt  = well_data["h_tvt_true"]
    t_gr   = well_data["t_gr"]
    t_tvt  = well_data["t_tvt"]
    ps_idx = well_data["ps_idx"]

    ps_shifted     = ps_idx + offset * step
    margin_before  = H_BEFORE_PS * step
    margin_after   = H_AFTER_PS  * step

    if ps_shifted < margin_before or ps_shifted + margin_after > len(h_gr):
        raise ValueError(
            f"offset={offset} places PS out of bounds "
            f"(ps_shifted={ps_shifted}, len={len(h_gr)})"
        )

    # ── Crop and compress lateral ─────────────────────────────────────────
    i0 = ps_shifted - margin_before
    i1 = ps_shifted + margin_after
    h_seg_gr  = h_gr[i0:i1].reshape(H_TOTAL, step).mean(axis=1)
    h_seg_tvt = h_tvt[i0:i1].reshape(H_TOTAL, step).mean(axis=1)

    # ── Typewell anchor ───────────────────────────────────────────────────
    ps_tvt = h_tvt[ps_shifted]
    t_ps   = int(np.abs(t_tvt - ps_tvt).argmin())
    j0, j1 = t_ps - T_HALF, t_ps + T_HALF

    if j0 < 0 or j1 > len(t_gr):
        pad_l = max(0, -j0)
        pad_r = max(0, j1 - len(t_gr))
        t_seg_gr  = np.pad(t_gr[max(0,j0):min(len(t_gr),j1)],
                           (pad_l, pad_r), mode="edge")
        t_seg_tvt = np.pad(t_tvt[max(0,j0):min(len(t_tvt),j1)],
                           (pad_l, pad_r), mode="edge")
    else:
        t_seg_gr  = t_gr[j0:j1]
        t_seg_tvt = t_tvt[j0:j1]

    # ── Heatmap H(z, x) = f(z) - g(x) ───────────────────────────────────
    heatmap = (t_seg_gr[:, None] - h_seg_gr[None, :]).astype(np.float32)

    # ── Ground truth boundary ─────────────────────────────────────────────
    # matched[col] = typewell row index closest to the lateral's true TVT
    matched = np.abs(
        t_seg_tvt[:, None] - h_seg_tvt[None, :]
    ).argmin(axis=0).astype(np.int32)

    # ── History image ─────────────────────────────────────────────────────
    history = np.zeros((T_TOTAL, H_TOTAL), dtype=np.float32)
    for col in range(H_BEFORE_PS - 1):
        cv2.line(
            history,
            (col,     int(matched[col])),
            (col + 1, int(matched[col + 1])),
            color=1.0, thickness=1, lineType=cv2.LINE_AA
        )

    # ── SDF ───────────────────────────────────────────────────────────────
    boundary_mask = np.zeros((T_TOTAL, H_TOTAL), dtype=np.uint8)
    for col in range(H_TOTAL - 1):
        cv2.line(
            boundary_mask,
            (col,     int(matched[col])),
            (col + 1, int(matched[col + 1])),
            color=1, thickness=1, lineType=cv2.LINE_AA
        )
    unsigned_dist = distance_transform_edt(
        1 - boundary_mask
    ).astype(np.float32)
    row_indices   = np.arange(T_TOTAL)[:, None]
    sign          = np.sign(row_indices - matched[None, :].astype(np.float32))
    sdf           = (sign * unsigned_dist).astype(np.float32)

    # ── Validity masks ────────────────────────────────────────────────────
    h_mask = np.ones(H_TOTAL, dtype=np.float32)
    if i0 < 0:
        h_mask[:int(np.ceil(-i0 / step))] = 0.0
    if i1 > len(h_gr):
        h_mask[H_TOTAL - int(np.ceil((i1 - len(h_gr)) / step)):] = 0.0

    t_mask = np.ones(T_TOTAL, dtype=np.float32)
    if j0 < 0:
        t_mask[:-j0] = 0.0
    if j1 > len(t_gr):
        t_mask[T_TOTAL - (j1 - len(t_gr)):] = 0.0

    return {
        "heatmap" : heatmap,
        "history" : history,
        "sdf"     : sdf,
        "matched" : matched.astype(np.float32),
        "label"   : matched.astype(np.float32) / T_TOTAL,
        "target"  : h_seg_tvt.astype(np.float32),
        "h_gr"    : h_seg_gr.astype(np.float32),
        "t_gr"    : t_seg_gr.astype(np.float32),
        "h_mask"  : h_mask,
        "t_mask"  : t_mask,
    }


# ─────────────────────────────────────────────────────────────────────────────
# PyTorch Dataset
# ─────────────────────────────────────────────────────────────────────────────

class GeoSteerDataset(Dataset):
    def __init__(self,
                 well_ids : list,
                 offsets  : list,
                 step     : int  = S,
                 split    : str  = "train",
                 verbose  : bool = True):
        self.step    = step
        self.offsets = offsets
        self.wells   = {}
        self.pairs   = []

        for wid in well_ids:
            try:
                wd = load_well_data(wid, split=split)
                self.wells[wid] = wd
                for off in offsets:
                    try:
                        build_sample(wd, offset=off, step=step)
                        self.pairs.append((wid, off))
                    except ValueError:
                        pass
            except Exception as e:
                if verbose:
                    print(f"  ⚠  Skipping {wid}: {e}")

        if verbose:
            print(f"  Loaded {len(self.pairs)} samples from {len(self.wells)} wells")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        wid, offset = self.pairs[idx]
        s = build_sample(self.wells[wid], offset=offset, step=self.step)
        return {
            "heatmap" : torch.from_numpy(s["heatmap"]).unsqueeze(0),
            "history" : torch.from_numpy(s["history"]).unsqueeze(0),
            "sdf"     : torch.from_numpy(s["sdf"]).unsqueeze(0),
            "label"   : torch.from_numpy(s["label"]),
            "target"  : torch.from_numpy(s["target"]),
            "h_gr"    : torch.from_numpy(s["h_gr"]),
            "t_gr"    : torch.from_numpy(s["t_gr"]),
            "h_mask"  : torch.from_numpy(s["h_mask"]),
            "t_mask"  : torch.from_numpy(s["t_mask"]),
        }


# ─────────────────────────────────────────────────────────────────────────────
# DataLoader factory — identical signature to the previous version
# so engine.py needs no changes to this call site
# ─────────────────────────────────────────────────────────────────────────────

def make_loader(dataset   : Dataset,
                batch_size: int,
                shuffle   : bool,
                rank      : int,
                world_size: int,
                num_workers: int = 2) -> tuple:
    """
    Build a DataLoader with DistributedSampler.
    Returns (loader, sampler, original_size) — same contract as before.
    """
    original_size = len(dataset)
    sampler = DistributedSampler(
        dataset,
        num_replicas = world_size,
        rank         = rank,
        shuffle      = shuffle,
        drop_last    = shuffle,
    )
    loader = DataLoader(
        dataset,
        batch_size   = batch_size,
        sampler      = sampler,
        num_workers  = num_workers,
        pin_memory   = True,
        persistent_workers = (num_workers > 0),
    )
    return loader, sampler, original_size

# Model

In [ ]:
%%writefile /kaggle/working/src/model_sdf.py
"""
src/model_sdf.py
================
GeoSteerNet: U-Net style encoder-decoder that predicts a Signed Distance
Function (SDF) from a (heatmap, history) image pair.

Includes a self-contained verification block at the bottom that checks
both the model and dataset.py are working correctly together.
"""

import sys
import os
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.config import CFG
from src.dataset import (
    GeoSteerDataset, load_well_data, build_sample,
    T_TOTAL, H_TOTAL, H_BEFORE_PS, S,
)


# ─────────────────────────────────────────────────────────────────────────────
# Building blocks
# ─────────────────────────────────────────────────────────────────────────────

class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size,
                      padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBnRelu(in_ch,  out_ch)
        self.conv2 = ConvBnRelu(out_ch, out_ch)
        self.proj  = (nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
                      if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        return F.relu(self.conv2(self.conv1(x)) + self.proj(x), inplace=True)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.res  = ResidualBlock(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        skip = self.res(x)
        return skip, self.pool(skip)


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.res = ResidualBlock(in_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:],
                          mode="bilinear", align_corners=False)
        return self.res(torch.cat([x, skip], dim=1))


# ─────────────────────────────────────────────────────────────────────────────
# GeoSteerNet
# ─────────────────────────────────────────────────────────────────────────────

class GeoSteerNet(nn.Module):
    """
    Input  : batch dict with 'heatmap' (B,1,64,24) and 'history' (B,1,64,24)
    Output : dict with 'sdf' (B,1,64,24) and 'sdf_loss' (scalar)
    """

    def __init__(self, base_ch: int = 32):
        super().__init__()
        b = base_ch

        self.stem = ConvBnRelu(2, b, kernel_size=5, padding=2)

        # Encoder: (64,24)→(32,12)→(16,6)→(8,3)
        self.down1 = DownBlock(b,     b * 2)
        self.down2 = DownBlock(b * 2, b * 4)
        self.down3 = DownBlock(b * 4, b * 8)

        # Bottleneck at (8,3)
        self.bottleneck = nn.Sequential(
            ResidualBlock(b * 8, b * 8),
            ResidualBlock(b * 8, b * 8),
        )

        # Decoder
        self.up3 = UpBlock(b * 8 + b * 8, b * 4)
        self.up2 = UpBlock(b * 4 + b * 4, b * 2)
        self.up1 = UpBlock(b * 2 + b * 2, b)

        # SDF head — no activation, unbounded output
        self.sdf_head = nn.Conv2d(b, 1, kernel_size=1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        nn.init.normal_(self.sdf_head.weight, std=0.01)
        nn.init.zeros_(self.sdf_head.bias)

    def forward(self, batch: dict) -> dict:
        x = torch.cat([batch["heatmap"], batch["history"]], dim=1)

        x = self.stem(x)

        skip1, x = self.down1(x)
        skip2, x = self.down2(x)
        skip3, x = self.down3(x)

        x = self.bottleneck(x)

        x = self.up3(x, skip3)
        x = self.up2(x, skip2)
        x = self.up1(x, skip1)

        sdf_pred = self.sdf_head(x)

        out = {"sdf": sdf_pred}
        if "sdf" in batch:
            out["sdf_loss"] = self._loss(
                sdf_pred, batch["sdf"],
                batch["h_mask"], batch["t_mask"]
            )
        return out

    def _loss(self, sdf_pred, sdf_true, h_mask, t_mask):
        """
        Proximity-weighted masked MSE.
        Cells near the boundary receive higher weight so the model
        focuses on getting the zero-crossing right.
        """
        h_mask_2d = h_mask[:, None, None, :]   # (B,1,1,H)
        t_mask_2d = t_mask[:, None, :, None]   # (B,1,T,1)
        mask_2d   = h_mask_2d * t_mask_2d      # (B,1,T,H)

        proximity = torch.exp(-torch.abs(sdf_true) / 5.0)
        weight    = mask_2d * proximity

        loss = (weight * (sdf_pred - sdf_true) ** 2).sum()
        loss = loss / (weight.sum() + 1e-8)
        return loss


def build_sdf_model(rank: int, base_ch: int = 32) -> GeoSteerNet:
    """Construct and move GeoSteerNet to the correct device."""
    device = torch.device(
        f"cuda:{rank}" if torch.cuda.is_available() else "cpu"
    )
    model = GeoSteerNet(base_ch=base_ch).to(device)
    n = sum(p.numel() for p in model.parameters())
    print(f"  GeoSteerNet: {n:,} parameters  (device: {device})")
    return model


# ─────────────────────────────────────────────────────────────────────────────
# Verification
# ─────────────────────────────────────────────────────────────────────────────

def verify_model_and_dataset(sample_id: str = "0dd99dc5"):
    """
    End-to-end check: dataset → batch → model → loss.
    Verifies that dataset.py and model_sdf.py are correctly wired together.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*55}")
    print(f"  Verification: dataset + model")
    print(f"  Device: {device}")
    print(f"{'='*55}\n")

    # ── Step 1: Build a small dataset from one well ───────────────────────
    print("STEP 1: Building GeoSteerDataset (1 well, offsets [-2,0,2])...")
    ds = GeoSteerDataset(
        well_ids = [sample_id],
        offsets  = [-2, 0, 2],
        split    = "train",
    )
    print(f"  Samples in dataset : {len(ds)}")
    assert len(ds) > 0, "Dataset is empty — check well_id and offsets"
    print("  ✓ Dataset non-empty\n")

    # ── Step 2: Check one raw sample from __getitem__ ─────────────────────
    print("STEP 2: Checking __getitem__ shapes...")
    sample = ds[0]
    expected = {
        "heatmap" : (1, T_TOTAL, H_TOTAL),
        "history" : (1, T_TOTAL, H_TOTAL),
        "sdf"     : (1, T_TOTAL, H_TOTAL),
        "label"   : (H_TOTAL,),
        "target"  : (H_TOTAL,),
        "h_gr"    : (H_TOTAL,),
        "t_gr"    : (T_TOTAL,),
        "h_mask"  : (H_TOTAL,),
        "t_mask"  : (T_TOTAL,),
    }
    all_ok = True
    for key, shape in expected.items():
        actual = tuple(sample[key].shape)
        status = "✓" if actual == shape else "✗"
        if actual != shape:
            all_ok = False
        print(f"  {status}  {key:<10} expected {str(shape):<18} got {actual}")
    assert all_ok, "Shape mismatch in __getitem__ output"
    print("  All shapes correct\n")

    # ── Step 3: Stack into a batch of 4 ──────────────────────────────────
    print("STEP 3: Building a batch of 4 via DataLoader...")
    loader = torch.utils.data.DataLoader(ds, batch_size=len(ds), shuffle=False)
    batch  = next(iter(loader))
    print(f"  heatmap batch shape : {tuple(batch['heatmap'].shape)}")
    print(f"  history batch shape : {tuple(batch['history'].shape)}")
    print(f"  sdf     batch shape : {tuple(batch['sdf'].shape)}")
    assert tuple(batch["heatmap"].shape) == (len(ds), 1, T_TOTAL, H_TOTAL), \
        "Unexpected heatmap batch shape"
    print("  ✓ Batch shapes correct\n")

    # ── Step 4: Forward pass through GeoSteerNet ─────────────────────────
    print("STEP 4: Forward pass through GeoSteerNet...")
    net = GeoSteerNet(base_ch=CFG.SDF_BASE_CH).to(device)
    n   = sum(p.numel() for p in net.parameters())
    print(f"  Parameters: {n:,}")

    batch_gpu = {k: v.to(device) for k, v in batch.items()
                 if isinstance(v, torch.Tensor)}

    with torch.no_grad():
        output = net(batch_gpu)

    print(f"  sdf output shape : {tuple(output['sdf'].shape)}")
    print(f"  sdf_loss         : {output['sdf_loss'].item():.4f}")

    assert tuple(output["sdf"].shape) == (len(ds), 1, T_TOTAL, H_TOTAL), \
        "Unexpected SDF output shape"
    assert torch.isfinite(output["sdf_loss"]), \
        "Loss is not finite"
    print("  ✓ Output shape correct")
    print("  ✓ Loss is finite\n")

    # ── Step 5: Backward pass ─────────────────────────────────────────────
    print("STEP 5: Backward pass (gradient check)...")
    net.train()
    output = net(batch_gpu)
    output["sdf_loss"].backward()

    # Check that at least one parameter received a gradient
    grads_ok = all(
        p.grad is not None and torch.isfinite(p.grad).all()
        for p in net.parameters() if p.requires_grad
    )
    assert grads_ok, "Some parameters have missing or non-finite gradients"
    print("  ✓ All gradients finite\n")

    print(f"{'='*55}")
    print("  All checks passed — dataset and model are correctly wired.")
    print(f"{'='*55}\n")


if __name__ == "__main__":
    verify_model_and_dataset("0dd99dc5")

# Inference

In [ ]:
%%writefile /kaggle/working/inference.py
"""
inference.py
============
ROGII Wellbore Geology Prediction — SDF Inference Script

Sliding window strategy: the prediction zone is covered by overlapping
windows of 16 compressed segments (512 ft each). Each window's 8-segment
history is drawn from the previous window's predictions, so the model
autoregressively tracks the boundary across the full lateral.

Launch from a Kaggle notebook cell:
    !python inference.py
"""

# ── Path guard ─────────────────────────────────────────────────────────────
import sys
import os
_WORKING_DIR = "/kaggle/working"
if _WORKING_DIR not in sys.path:
    sys.path.insert(0, _WORKING_DIR)

# ── Standard library ───────────────────────────────────────────────────────
import gc
import ctypes
from pathlib import Path
from threading import Thread
from collections import defaultdict
from tqdm import tqdm

# ── Third-party ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch

# ── Project modules ────────────────────────────────────────────────────────
from src.config import CFG, RUN_VERSION
from src.dataset import (
    load_well_data, build_sample,
    S, H_BEFORE_PS, H_AFTER_PS, H_TOTAL, T_TOTAL, T_HALF,
)
from src.model_sdf import GeoSteerNet

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

CHECKPOINT_DIR = "/kaggle/input/notebooks/medali1992/rogii-cnn-mtp-train/checkpoints/"
KAGGLE_DIR     = CFG.DATA_ROOT


# ─────────────────────────────────────────────────────────────────────────────
# MEMORY CLEANUP
# ─────────────────────────────────────────────────────────────────────────────

def clean_memory(deep: bool = True) -> None:
    gc.collect()
    if deep:
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass
    torch.cuda.empty_cache()


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: FIND CHECKPOINTS
# ─────────────────────────────────────────────────────────────────────────────

def find_checkpoints(checkpoint_dir: str, version: str) -> list:
    """
    Find the best checkpoint per fold for the given RUN_VERSION.

    Filename format: sdf_v{VERSION}_fold{N}_ep{E}_rmse{R}_{run}.pth
    We keep one checkpoint per fold — the one with the lowest val_rmse
    encoded in the filename.

    Returns a list of Path objects sorted by fold index.
    """
    ckpt_dir  = Path(checkpoint_dir)
    all_ckpts = sorted(ckpt_dir.glob(f"sdf_v{version}_fold*.pth"))

    if len(all_ckpts) == 0:
        raise FileNotFoundError(
            f"No SDF checkpoints found for v{version} in {checkpoint_dir}.\n"
            f"Files present: {list(ckpt_dir.glob('*.pth'))}"
        )

    # Group by fold, keep lowest rmse per fold
    fold_map = defaultdict(list)
    for p in all_ckpts:
        parts     = p.stem.split("_")
        fold_part = next((x for x in parts if x.startswith("fold")), None)
        rmse_part = next((x for x in parts if x.startswith("rmse")), None)
        if fold_part is None or rmse_part is None:
            continue
        fold_idx = int(fold_part.replace("fold", ""))
        rmse_val = float(rmse_part.replace("rmse", ""))
        fold_map[fold_idx].append((rmse_val, p))

    best_per_fold = []
    for fold_idx in sorted(fold_map.keys()):
        best_rmse, best_path = min(fold_map[fold_idx], key=lambda x: x[0])
        best_per_fold.append(best_path)
        print(f"  fold {fold_idx}: {best_path.name}  "
              f"(val_rmse={best_rmse:.4f})")

    return best_per_fold

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: BUILD ONE INFERENCE WINDOW
# ─────────────────────────────────────────────────────────────────────────────

def build_inference_window(well_data      : dict,
                           ps_shifted     : int,
                           pred_tvt_array : np.ndarray,
                           step           : int = S) -> dict:
    """
    Build (heatmap, history) for one sliding window position.

    Identical to build_sample except the history image is constructed
    from pred_tvt_array rather than ground truth h_tvt_true.

    pred_tvt_array : shape (len(h_gr),) float32
        TVT value at every 1-ft lateral position.
        Before the true PS  → filled from h_tvt_true (ground truth).
        After  the true PS  → filled from previous window predictions.
        Positions not yet predicted → np.nan (masked out of history).

    Returns dict with keys: heatmap, history, h_mask, t_mask,
                             t_seg_tvt, h_seg_gr
    (t_seg_tvt is returned so rows_to_tvt can convert row indices → TVT)
    """
    h_gr   = well_data["h_gr_smooth"]
    t_gr   = well_data["t_gr"]
    t_tvt  = well_data["t_tvt"]

    margin_before = H_BEFORE_PS * step
    margin_after  = H_AFTER_PS  * step

    # Safety check
    if ps_shifted < margin_before or ps_shifted + margin_after > len(h_gr):
        raise ValueError(
            f"ps_shifted={ps_shifted} out of bounds "
            f"(need [{margin_before}, {len(h_gr) - margin_after}])"
        )

    # ── Crop and compress lateral ─────────────────────────────────────────
    i0 = ps_shifted - margin_before
    i1 = ps_shifted + margin_after
    h_seg_gr = h_gr[i0:i1].reshape(H_TOTAL, step).mean(axis=1)  # (24,)

    # ── Typewell anchor at this window's PS TVT ───────────────────────────
    # Use pred_tvt_array at ps_shifted as anchor — this is always filled
    # (either ground truth or prediction from previous window)
    ps_tvt = pred_tvt_array[ps_shifted]
    if np.isnan(ps_tvt):
        # Fallback to true TVT if prediction somehow missing
        ps_tvt = well_data["h_tvt_true"][ps_shifted]

    t_ps   = int(np.abs(t_tvt - ps_tvt).argmin())
    j0, j1 = t_ps - T_HALF, t_ps + T_HALF

    if j0 < 0 or j1 > len(t_gr):
        pad_l = max(0, -j0)
        pad_r = max(0, j1 - len(t_gr))
        t_seg_gr  = np.pad(t_gr[max(0,j0):min(len(t_gr),j1)],
                           (pad_l, pad_r), mode="edge")
        t_seg_tvt = np.pad(t_tvt[max(0,j0):min(len(t_tvt),j1)],
                           (pad_l, pad_r), mode="edge")
    else:
        t_seg_gr  = t_gr[j0:j1]
        t_seg_tvt = t_tvt[j0:j1]

    # ── Heatmap ───────────────────────────────────────────────────────────
    heatmap = (t_seg_gr[:, None] - h_seg_gr[None, :]).astype(np.float32)

    # ── History image from pred_tvt_array ─────────────────────────────────
    # For each history column (0 to H_BEFORE_PS-1), find which typewell
    # row corresponds to the predicted TVT at that lateral position.
    # If TVT is NaN (not yet predicted), skip that column.
    import cv2
    history = np.zeros((T_TOTAL, H_TOTAL), dtype=np.float32)

    hist_matched = np.full(H_BEFORE_PS, T_HALF, dtype=np.float32)  # default: center row
    for col in range(H_BEFORE_PS):
        # 1-ft index at the center of this compressed segment
        ft_idx = i0 + col * step + step // 2
        tvt_val = pred_tvt_array[ft_idx]
        if np.isnan(tvt_val):
            continue
        # Find closest typewell row to this TVT value
        row = int(np.abs(t_seg_tvt - tvt_val).argmin())
        hist_matched[col] = row

    # Draw history line only for valid pre-PS columns
    for col in range(H_BEFORE_PS - 1):
        cv2.line(
            history,
            (col,     int(hist_matched[col])),
            (col + 1, int(hist_matched[col + 1])),
            color=1.0, thickness=1, lineType=cv2.LINE_AA,
        )

    # ── Validity masks ────────────────────────────────────────────────────
    h_mask = np.ones(H_TOTAL, dtype=np.float32)
    if i0 < 0:
        h_mask[:int(np.ceil(-i0 / step))] = 0.0
    if i1 > len(h_gr):
        h_mask[H_TOTAL - int(np.ceil((i1 - len(h_gr)) / step)):] = 0.0

    t_mask = np.ones(T_TOTAL, dtype=np.float32)
    if j0 < 0:
        t_mask[:-j0] = 0.0
    if j1 > len(t_gr):
        t_mask[T_TOTAL - (j1 - len(t_gr)):] = 0.0

    return {
        "heatmap"   : heatmap,
        "history"   : history,
        "h_mask"    : h_mask,
        "t_mask"    : t_mask,
        "t_seg_tvt" : t_seg_tvt,   # needed by rows_to_tvt
        "h_seg_gr"  : h_seg_gr,    # useful for debugging
    }

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: CONVERT BOUNDARY ROWS → TVT VALUES
# ─────────────────────────────────────────────────────────────────────────────

def rows_to_tvt(pred_rows  : np.ndarray,
                t_seg_tvt  : np.ndarray) -> np.ndarray:
    """
    Convert predicted boundary row indices → TVT values.

    pred_rows  : (H_TOTAL,) int   — argmin(|SDF|) per lateral column
    t_seg_tvt  : (T_TOTAL,) float — typewell TVT for this window's crop

    The model predicts which typewell row (0–63) the boundary sits at
    for each lateral segment. We simply look up the TVT value at that
    row in the local typewell crop.

    Rows are clipped to [0, T_TOTAL-1] as a safety guard against any
    edge cases where argmin returns an out-of-range index.

    Returns: (H_TOTAL,) float32 — TVT prediction per lateral segment
    """
    rows_clipped = np.clip(pred_rows.astype(int), 0, T_TOTAL - 1)
    return t_seg_tvt[rows_clipped].astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: SLIDING WINDOW PREDICTION FOR ONE WELL
# ─────────────────────────────────────────────────────────────────────────────

def sliding_window_predict(well_data : dict,
                            models   : list,
                            device   : str,
                            step     : int = S) -> np.ndarray:
    """
    Predict TVT for the full prediction zone of one well using a
    sliding window strategy.

    The prediction zone starts at ps_idx+1 and runs to the end of
    the well. We slide the window forward by H_AFTER_PS segments
    (512 ft) at each step. Each window's history cols are filled
    from previously predicted TVT values.

    models : list of GeoSteerNet instances (one per fold).
             Their predictions are averaged for ensemble inference.

    Returns: pred_tvt_array : (len(h_gr),) float32
        TVT filled for all positions up to and including the last
        predicted segment. Positions before ps_idx contain ground
        truth. Positions not reached by any window contain NaN.
    """
    h_gr     = well_data["h_gr_smooth"]
    h_tvt    = well_data["h_tvt_true"]
    ps_idx   = well_data["ps_idx"]

    # Initialise pred_tvt_array:
    # pre-PS  → ground truth TVT
    # post-PS → NaN (filled as we slide forward)
    pred_tvt_array = np.full(len(h_gr), np.nan, dtype=np.float32)
    pred_tvt_array[:ps_idx + 1] = h_tvt[:ps_idx + 1]

    margin_before = H_BEFORE_PS * step   # 256 ft
    margin_after  = H_AFTER_PS  * step   # 512 ft

    # First window is anchored at ps_idx
    # Slide forward by H_AFTER_PS segments each time
    ps_shifted = ps_idx

    while True:
        # Check we have enough room for this window
        if ps_shifted + margin_after > len(h_gr):
            # Last partial window — anchor as far back as possible
            ps_shifted = len(h_gr) - margin_after
            if ps_shifted < margin_before:
                break  # well is too short for even one window

        # ── Build window ──────────────────────────────────────────────
        try:
            window = build_inference_window(
                well_data, ps_shifted, pred_tvt_array, step=step
            )
        except ValueError:
            break

        # ── Prepare batch tensors ─────────────────────────────────────
        heatmap_t = torch.from_numpy(
            window["heatmap"][None, None, :, :]   # (1,1,64,24)
        ).float().to(device)

        history_t = torch.from_numpy(
            window["history"][None, None, :, :]   # (1,1,64,24)
        ).float().to(device)

        batch_gpu = {"heatmap": heatmap_t, "history": history_t}

        # ── Ensemble prediction (average across folds) ────────────────
        row_preds = np.zeros(H_TOTAL, dtype=np.float32)

        with torch.no_grad():
            for model in models:
                sdf_pred  = model(batch_gpu)["sdf"]          # (1,1,64,24)
                rows      = torch.abs(
                    sdf_pred[:, 0, :, :]
                ).argmin(dim=1)[0].cpu().numpy()              # (24,)
                row_preds += rows.astype(np.float32)

        row_preds /= len(models)
        row_preds  = np.round(row_preds).astype(int)

        # ── Convert rows → TVT ────────────────────────────────────────
        tvt_preds = rows_to_tvt(row_preds, window["t_seg_tvt"])

        # ── Write predictions into pred_tvt_array ─────────────────────
        # Only write the H_AFTER_PS prediction columns (cols 8–23)
        # History columns (0–7) are already filled
        pred_start = ps_shifted               # 1-ft index of col 8
        for col in range(H_AFTER_PS):
            ft_start = ps_shifted + col * step
            ft_end   = min(ft_start + step, len(h_gr))
            # Never overwrite the known pre-PS zone
            ft_start = max(ft_start, ps_idx + 1)
            if ft_start >= ft_end or ft_start >= len(h_gr):
                break
            pred_tvt_array[ft_start:ft_end] = tvt_preds[
                H_BEFORE_PS + col
            ]

        # ── Advance window ────────────────────────────────────────────
        # Fill the next window's anchor position before advancing.
        # ps_shifted + margin_after is the anchor for the next window —
        # it must not be NaN or the typewell re-centering breaks.
        next_anchor = ps_shifted + margin_after
        if next_anchor < len(h_gr):
            # Use the last filled position's TVT as the anchor
            pred_tvt_array[next_anchor] = tvt_preds[H_BEFORE_PS + H_AFTER_PS - 1]

        # Stop if we've covered the full well
        if ps_shifted >= len(h_gr):
            break

    return pred_tvt_array

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: SINGLE-GPU INFERENCE (single-window, no sliding)
# ─────────────────────────────────────────────────────────────────────────────

def run_inference_on_device(well_ids     : list,
                             ckpt_paths  : list,
                             device      : str,
                             results_dict: dict,
                             split       : str = "test") -> None:
    """
    Single-window fold-ensemble inference on one GPU.

    One forward pass per well at PS — no sliding window.
    The 16-segment (512 ft) prediction is written post-PS.
    Everything beyond 512 ft is filled with the last predicted value.

    This is simpler and faster than sliding window, and lets us
    validate whether the TVT conversion is correct before adding
    complexity back.
    """
    # ── Load all fold models ──────────────────────────────────────────────
    print(f"  [{device}] Loading {len(ckpt_paths)} fold checkpoints...")
    models = []
    for ckpt_path in ckpt_paths:
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        net  = GeoSteerNet(
            base_ch=ckpt.get("cfg", {}).get("SDF_BASE_CH", CFG.SDF_BASE_CH)
        ).to(device)
        net.load_state_dict(ckpt["model_state"])
        net.eval()
        models.append(net)
        print(f"  [{device}]   loaded: {ckpt_path.name}")

    # ── Iterate wells ─────────────────────────────────────────────────────
    device_results = {}

    for well_id in tqdm(well_ids, desc=f"[{device}]", leave=True):
        try:
            wd  = load_well_data(well_id, split=split)
            ps  = wd["ps_idx"]
            n   = len(wd["h_gr_smooth"])

            # Initialise: pre-PS from ground truth, post-PS = NaN
            pred_tvt_array = np.full(n, np.nan, dtype=np.float32)
            pred_tvt_array[:ps + 1] = wd["h_tvt_true"][:ps + 1]

            # ── Single forward pass at PS ─────────────────────────────
            window = build_inference_window(
                wd, ps, pred_tvt_array, step=S
            )

            heatmap_t = torch.from_numpy(
                window["heatmap"][None, None]
            ).float().to(device)
            history_t = torch.from_numpy(
                window["history"][None, None]
            ).float().to(device)
            batch_gpu = {"heatmap": heatmap_t, "history": history_t}

            # Ensemble: average row predictions across folds
            row_preds = np.zeros(H_TOTAL, dtype=np.float32)
            with torch.no_grad():
                for model in models:
                    sdf  = model(batch_gpu)["sdf"]        # (1,1,64,24)
                    rows = torch.abs(
                        sdf[:, 0]
                    ).argmin(dim=1)[0].cpu().numpy()       # (24,)
                    row_preds += rows.astype(np.float32)

            row_preds = np.round(row_preds / len(models)).astype(int)

            # Convert rows → TVT using local typewell crop
            tvt_preds = rows_to_tvt(row_preds, window["t_seg_tvt"])

            # ── Write 16 post-PS segments into pred_tvt_array ────────
            for col in range(H_AFTER_PS):
                ft_start = max(ps + col * S + 1, ps + 1)
                ft_end   = min(ft_start + S, n)
                if ft_start >= n:
                    break
                pred_tvt_array[ft_start:ft_end] = tvt_preds[
                    H_BEFORE_PS + col
                ]

            # ── Fill remainder beyond 512 ft with last prediction ────
            last_valid_idx = np.where(~np.isnan(pred_tvt_array))[0]
            if len(last_valid_idx) > 0:
                last_val = pred_tvt_array[last_valid_idx[-1]]
                pred_tvt_array[np.isnan(pred_tvt_array)] = last_val

            device_results[well_id] = pred_tvt_array

        except Exception as e:
            print(f"  [{device}] ⚠  {well_id} failed: {e}")
            try:
                wd         = load_well_data(well_id, split=split)
                tvt        = wd["h_tvt_true"].copy()
                last_known = tvt[wd["ps_idx"]]
                tvt[wd["ps_idx"] + 1:] = last_known
                device_results[well_id] = tvt
            except Exception as e2:
                print(f"  [{device}] ⚠  {well_id} fallback failed: {e2}")
                device_results[well_id] = None

    results_dict[device] = device_results
    del models
    clean_memory(deep=True)
    print(f"  [{device}] Done — {len(device_results)} wells predicted.")

# ─────────────────────────────────────────────────────────────────────────────
# MAIN INFERENCE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 65)
    print(f"  ROGII Wellbore Geology — SDF Inference v{RUN_VERSION}")
    print("=" * 65)

    # ── 1. Check available GPUs ───────────────────────────────────────────
    n_gpus  = torch.cuda.device_count()
    devices = [f"cuda:{i}" for i in range(n_gpus)] if n_gpus > 0 else ["cpu"]
    print(f"\n  Available GPUs : {n_gpus}")
    for d in devices:
        if d != "cpu":
            idx = int(d.split(":")[1])
            print(f"    {d}  {torch.cuda.get_device_name(idx)}")

    # ── 2. Find checkpoints ───────────────────────────────────────────────
    print(f"\n  Loading checkpoints from {CHECKPOINT_DIR}:")
    ckpt_paths = find_checkpoints(CHECKPOINT_DIR, RUN_VERSION)
    print(f"  {len(ckpt_paths)} fold checkpoints found.")

    # ── 3. Find test wells ────────────────────────────────────────────────
    test_dir   = f"{KAGGLE_DIR}/test"
    test_ids   = sorted(set(
        f.replace("__horizontal_well.csv", "")
        for f in os.listdir(test_dir)
        if f.endswith("__horizontal_well.csv")
    ))
    print(f"\n  Test wells: {len(test_ids)}")

    # ── 4. Load submission template ───────────────────────────────────────
    sample_sub = pd.read_csv(f"{KAGGLE_DIR}/sample_submission.csv")
    print(f"  Submission rows: {len(sample_sub):,}")

    # Parse well_id and row_index from submission id column
    # Format: "{well_id}_{row_index}"
    split_ids             = sample_sub["id"].str.rsplit("_", n=1, expand=True)
    sample_sub["well_id"] = split_ids[0]
    sample_sub["row_idx"] = split_ids[1].astype(int)

    # ── 5. Split wells across GPUs ────────────────────────────────────────
    # Round-robin by well index so both GPUs get similar workloads
    seqs_per_device = {dev: [] for dev in devices}
    for i, wid in enumerate(test_ids):
        seqs_per_device[devices[i % len(devices)]].append(wid)

    for dev in devices:
        print(f"  {dev}: {len(seqs_per_device[dev])} wells")

    # ── 6. Threaded inference ─────────────────────────────────────────────
    print("\n  Running inference...")
    results = {}
    threads = []

    for dev in devices:
        t = Thread(
            target = run_inference_on_device,
            args   = (seqs_per_device[dev], ckpt_paths,
                      dev, results, "test"),
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print("\n  All threads complete.")

    # ── 7. Merge results from all GPUs ────────────────────────────────────
    merged = {}
    for dev in devices:
        if results.get(dev):
            merged.update(results[dev])

    print(f"  Total wells predicted: {len(merged)}")

    # ── 8. Build submission CSV ───────────────────────────────────────────
    print("\n  Building submission...")

    def lookup_tvt(row):
        """Look up predicted TVT for one submission row."""
        wid     = row["well_id"]
        row_idx = int(row["row_idx"])
        arr     = merged.get(wid)
        if arr is None or row_idx >= len(arr):
            return np.nan
        val = arr[row_idx]
        return float(val) if not np.isnan(val) else np.nan

    sample_sub["tvt"] = sample_sub.apply(lookup_tvt, axis=1)

    # Fill any remaining NaN with per-well last-known TVT
    nan_mask = sample_sub["tvt"].isna()
    if nan_mask.sum() > 0:
        print(f"  ⚠  {nan_mask.sum()} NaN predictions — filling with "
              f"last known TVT per well")
        for wid, grp in sample_sub[nan_mask].groupby("well_id"):
            arr = merged.get(wid)
            if arr is not None:
                last_known = arr[~np.isnan(arr)][-1] \
                             if np.any(~np.isnan(arr)) else 0.0
            else:
                last_known = 0.0
            sample_sub.loc[grp.index, "tvt"] = last_known

    # ── 9. Sanity checks ──────────────────────────────────────────────────
    pred_values = sample_sub["tvt"].values
    print(f"\n  Prediction statistics:")
    print(f"    n rows : {len(pred_values):,}")
    print(f"    mean   : {pred_values.mean():.2f}")
    print(f"    std    : {pred_values.std():.2f}")
    print(f"    min    : {pred_values.min():.2f}")
    print(f"    max    : {pred_values.max():.2f}")
    print(f"    NaN    : {np.isnan(pred_values).sum()}")

    assert np.isnan(pred_values).sum() == 0, "NaN values remain in submission"
    assert pred_values.std() > 1.0,          "Predictions have near-zero variance"

    # ── 10. Write submission ──────────────────────────────────────────────
    sub_path = "/kaggle/working/submission.csv"
    sample_sub[["id", "tvt"]].to_csv(sub_path, index=False)
    print(f"\n  submission.csv → {sub_path}")
    print(f"  Rows: {len(sample_sub):,}")
    print("=" * 65)

    clean_memory(deep=True)

In [ ]:
! python inference.py

# Sanity check

In [ ]:
import pandas as pd
import numpy as np

sub = pd.read_csv("/kaggle/working/submission.csv")

print(f"Shape          : {sub.shape}")
print(f"Columns        : {list(sub.columns)}")
print(f"NaN count      : {sub['tvt'].isna().sum()}")
print(f"Unique wells   : {sub['id'].str.rsplit('_', n=1).str[0].nunique()}")
print(f"\nFirst 5 rows:")
print(sub.head())
print(f"\nLast 5 rows:")
print(sub.tail())

# Check id format matches expected pattern
sample_ids = sub["id"].head(3).tolist()
print(f"\nSample IDs: {sample_ids}")

# Check TVT range is reasonable
assert sub["tvt"].min() > 5000,  "TVT too small — check units"
assert sub["tvt"].max() < 20000, "TVT too large — check units"
assert sub["tvt"].isna().sum() == 0, "NaN values in submission"
print("\n✓ Submission looks correct — ready to submit")